<a href="https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


The rule, in plain words: a page is worth checking first if it is young (under 180 days old), has not been updated for 90+ days, and is visible (100+ views in 90 days and a search position). The longer it has been left alone, the higher it ranks.

Reason code (only one): young_untouched_visible. Action: review_for_refresh. Every other page gets "none" and "monitor".

Signal check 1: days since last update (behind FlyRank's refresh flags). Verdict: MIXED.
The declining rate climbs from 0.52 (14-29 days, n=17,052) to 0.61 (90-179 days, n=9,171). But pages at 180+ days (n=174) are at 0.47, no worse than average, so the 180-day cut-off in the starter flag misses where the signal is. I use 90 days.

Signal check 2: page age (behind FlyRank's page-one decay flag, which assumes older pages decay). Verdict: OPPOSITE.
Younger pages decline more: 0.63 at 90-179 days (n=12,014), 0.52 at 180-364 days (n=11,626), 0.43 at 365+ days (n=6,360). So I do not rank old pages higher. I use young pages. This negative saved my rule from copying the wrong idea.

 I chose these thresholds by looking at this one file, so the rule's scores below are optimistic. In ML-08 I will score the rule and the model on the same held-out clients.

In [1]:
import os, sys, subprocess, json
import numpy as np, pandas as pd

# In Colab: download the repo once. On your computer: go to the repo folder.
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("Flyrank-MLE-intern"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ysl21292/Flyrank-MLE-intern.git"], check=True)
    os.chdir("Flyrank-MLE-intern")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declined"] = (df.trend_direction == "down").astype(int)   # used only to CHECK signals and score the rule
print(f"{len(df):,} pages | base rate (share declining): {df.declined.mean():.1%}\n")

def bucket_table(col, bins, labels):
    b = pd.cut(df[col], bins=bins, labels=labels, right=False)
    t = df.groupby(b, observed=True).declined.agg(n="size", declining_rate="mean")
    t["declining_rate"] = t.declining_rate.round(2)
    return t

print("Signal 1 - days since last update (staleness, behind FlyRank's refresh flags)")
print(bucket_table("days_since_last_update", [0, 14, 30, 90, 180, 10000], ["0-13", "14-29", "30-89", "90-179", "180+"]))
print("\nSignal 2 - page age in days (behind FlyRank's page-one decay flag)")
print(bucket_table("content_age_days", [0, 180, 365, 10000], ["90-179", "180-364", "365+"]))

30,000 pages | base rate (share declining): 54.2%

Signal 1 - days since last update (staleness, behind FlyRank's refresh flags)
                            n  declining_rate
days_since_last_update                       
0-13                     3428            0.48
14-29                   17052            0.52
30-89                     175            0.59
90-179                   9171            0.61
180+                      174            0.47

Signal 2 - page age in days (behind FlyRank's page-one decay flag)
                      n  declining_rate
content_age_days                       
90-179            12014            0.63
180-364           11626            0.52
365+               6360            0.43


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


Score = days since last update, for flagged pages only (the longer left alone, the higher). Every other page scores 0. Ties are broken by a seeded random number, not by file order, because the file order is not random and would flatter the result.

The queue is saved to work/outputs/baseline_action_score.csv (not committed to git) and the results to work/outputs/baseline_metrics.json.

Result on this file (observed): Precision@20 = 0.85, Precision@50 = 0.86, Precision@100 = 0.82, against a base rate of 0.54 for random picking.

In [2]:
# The rule, in plain words in section 1. It reads observed columns only. The label is not used here.
young     = df.content_age_days < 180
untouched = df.days_since_last_update >= 90
visible   = (df.impressions_90d >= 100) & (df.avg_position > 0)   # avg_position 0 means "no data"
flagged   = young & untouched & visible

df["score"] = np.where(flagged, df.days_since_last_update, 0)      # the longer it was left alone, the higher
df["reason_code"] = np.where(flagged, "young_untouched_visible", "none")
df["action"] = np.where(flagged, "review_for_refresh", "monitor")

# Rank. Ties are broken by a seeded random number, NOT by file order (file order is not random).
df["tiebreak"] = np.random.default_rng(42).random(len(df))
queue = df.sort_values(["score", "tiebreak"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
        "days_since_last_update", "content_age_days", "impressions_90d", "avg_position", "ctr"]
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

def precision_at_k(k):
    return queue.declined.head(k).mean()

base = df.declined.mean()
results = {"base_rate": round(float(base), 3), "flagged_pages": int(flagged.sum()),
           "precision_at_20": round(float(precision_at_k(20)), 3),
           "precision_at_50": round(float(precision_at_k(50)), 3),
           "precision_at_100": round(float(precision_at_k(100)), 3)}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Flagged pages: {int(flagged.sum()):,} of {len(df):,}")
print(f"Base rate (random picking): {base:.2f}")
for k in (20, 50, 100):
    print(f"Precision@{k}: {precision_at_k(k):.2f}")
print("Saved work/outputs/baseline_action_score.csv and baseline_metrics.json")

Flagged pages: 1,944 of 30,000
Base rate (random picking): 0.54
Precision@20: 0.85
Precision@50: 0.86
Precision@100: 0.82
Saved work/outputs/baseline_action_score.csv and baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


Each line gives the action, the reason code, a confidence note, and what would make it wrong. Confidence is "higher" when a page has 1,000+ views and "lower" otherwise.

What I notice: all 20 pages sit at exactly 106 days. That is not 20 different stories. It looks like pages from one batch, so a human should check them together, not one by one.

In [3]:
def wrong_if(r):
    if r.impressions_90d < 500:
        return "it has under 500 views, so a small swing can look like a drop"
    if r.ctr == 0:
        return "it records no clicks, so the drop may be noise"
    if r.avg_position >= 30:
        return "it ranks deep (position 30+), so there is little to protect"
    return "the dip is seasonal, or a newer page took its traffic"

for _, r in queue.head(20).iterrows():
    conf = "higher" if r.impressions_90d >= 1000 else "lower"
    print(f"#{int(r['rank'])} {r.action} [{r.reason_code}], confidence {conf}: "
          f"{int(r.content_age_days)} days old, untouched {int(r.days_since_last_update)} days, "
          f"{int(r.impressions_90d):,} views at position {r.avg_position:.0f}. "
          f"Wrong if {wrong_if(r)}.")

#1 review_for_refresh [young_untouched_visible], confidence higher: 106 days old, untouched 106 days, 1,252 views at position 6. Wrong if the dip is seasonal, or a newer page took its traffic.
#2 review_for_refresh [young_untouched_visible], confidence higher: 106 days old, untouched 106 days, 1,266 views at position 5. Wrong if it records no clicks, so the drop may be noise.
#3 review_for_refresh [young_untouched_visible], confidence higher: 106 days old, untouched 106 days, 10,910 views at position 38. Wrong if it ranks deep (position 30+), so there is little to protect.
#4 review_for_refresh [young_untouched_visible], confidence higher: 106 days old, untouched 106 days, 1,032 views at position 11. Wrong if it records no clicks, so the drop may be noise.
#5 review_for_refresh [young_untouched_visible], confidence lower: 106 days old, untouched 106 days, 933 views at position 13. Wrong if the dip is seasonal, or a newer page took its traffic.
#6 review_for_refresh [young_untouched_vis

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Weak picks: 7 of the top 50 are not declining (for example ranks 2, 7, 12, 27 and 28). Ranks 7 and 28 sit at position 29-31, deep in the results, and rank 2 gets no clicks, which matches the "wrong if" notes above.

A bigger caution: 1,849 of the 1,944 flagged pages (95%) have exactly 104 days since the last update, and the top 74 pages have exactly 106. That looks like a batch update, not page-by-page neglect. Those 74 pages decline about 90% of the time and 72% of them come from one client; the top 3 clients hold 94% of all flagged pages. So the high precision may mostly say "this batch of pages from a few clients declined", not "untouched pages decline".

Leakage check: the rule reads only content_age_days, days_since_last_update, impressions_90d and avg_position. It never reads trend_direction, trend_pct, or the IDs, and it has no product flags or future windows. The label is used only to score the rule.

Claims: observed on this starter file, directional, decision-support. Not proof that updating pages helps.

In [4]:
top50 = queue.head(50)
weak = top50[top50.declined == 0]
print(f"Weak picks: {len(weak)} of the top 50 are NOT declining. First five:")
print(weak[["rank", "days_since_last_update", "content_age_days", "impressions_90d", "avg_position", "ctr"]].head(5).to_string(index=False))

# Leakage check: which columns does the rule read?
USED = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position"]
BANNED = ["trend_direction", "trend_pct", "content_id", "client_id"]
print("\nColumns the rule reads:", USED)
print("Any banned column used as an input?", any(c in USED for c in BANNED))

# Caution: where do the flagged pages come from?
share = df[flagged].client_id.value_counts(normalize=True)
print(f"\nFlagged pages come from {share.size} clients; the top 3 clients hold {share.head(3).sum():.0%} of them.")

Weak picks: 7 of the top 50 are NOT declining. First five:
 rank  days_since_last_update  content_age_days  impressions_90d  avg_position  ctr
    2                     106               106             1266           4.6 0.00
    7                     106               106              741          31.1 0.40
   12                     106               106             1755          20.2 0.40
   27                     106               106             1205           6.6 0.17
   28                     106               106              357          29.3 0.28

Columns the rule reads: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position']
Any banned column used as an input? False

Flagged pages come from 10 clients; the top 3 clients hold 94% of them.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.